In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import logging
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from egh_vlm.hallu_dataset import load_hallu_dataset, hallu_collate_fn
from egh_vlm.hallu_ffn_detector import HalluFFNDetector, train_ffn_detector, eval_ffn_detector
from egh_vlm.utils import create_balanced_dataloader, load_phd_dataset, split_stratified


In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/features_egh_full_context_phd_base_qwen3.pt'
DETECTOR_PATH = f'{prefix_path}/data/phd/detectors/ffn_egh_full_context_base_qwen3.pt'
OUTPUT_PATH = f'{prefix_path}/data/phd/evaluations/evaluation_egh_full_context_ffn_phd_base_qwen3.json'
LOGGING_PATH = f'{prefix_path}/data/logs/log_egh_full_context_ffn_phd_base_qwen3.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_hallu_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

hidden_size = features.embs[0].size(-1) if len(features) > 0 else 0
print(f'Hidden size of features: {hidden_size}')

In [ ]:
train_dataset, test_dataset = split_stratified(features, train_ratio=TRAIN_RATIO, random_state=42)
train_dataloader = create_balanced_dataloader(
    train_dataset,
    batch_size=32,
    collate_fn=hallu_collate_fn,
    shuffle=True,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=hallu_collate_fn,
    shuffle=True,
)

#### Experiment

In [ ]:
def train(
    train_dataloader: DataLoader,
    test_dataloader: DataLoader,
    hidden_size: int,
    logging_path: str,
    device: torch.device,
    train_ratio: float=0.7,
    epochs_count: int=30,
    lr: float=1e-4,
    features_weight: float=0.5,
):  
    logging.basicConfig(filename=logging_path, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    logging.info(f'Starting training with hidden_size={hidden_size}, train_ratio={train_ratio}, epochs_count={epochs_count}, lr={lr}, features_weight={features_weight}')
    
    result = {
        'train_ratio': train_ratio,
        'epochs_count': epochs_count,
        'lr': lr,
        'features_weight': features_weight,
        'acc': {
            'value': 0.0,
            'epoch': -1
        },
        'f1': {
            'value': 0.0,
            'epoch': -1
        },
        'pr_auc': {
            'value': 0.0,
            'epoch': -1
        }
    }
    detector = HalluFFNDetector(
        input_dim=hidden_size, 
        hidden_dim=hidden_size, 
        output_dim=1,
        device=device,
        w=features_weight)
    loss_fn = nn.BCELoss()
    optim = torch.optim.Adam(detector.parameters(), lr=lr)
    
    for i in range(epochs_count):
        loss = train_ffn_detector(detector, loss_fn, optim, train_dataloader)
        eval_metrics = eval_ffn_detector(detector, test_dataloader)
        acc = eval_metrics['acc']
        f1 = eval_metrics['f1']
        pr_auc = eval_metrics['pr_auc']
        
        logging.info(f'Epoch {i+1}/{epochs_count}, Loss: {loss:.4f}')
        logging.info(f'Epoch {i+1}/{epochs_count}, Accuracy: {acc:.4f}, F1: {f1:.4f}, PR-AUC: {pr_auc:.4f}\n')
        print(f'Epoch {i+1}/{epochs_count}, Loss: {loss:.4f}')
        print(f'Epoch {i+1}/{epochs_count}, Accuracy: {acc:.4f}, F1: {f1:.4f}, PR-AUC: {pr_auc:.4f}')

        if acc > result['acc']['value']:
            result['acc']['value'] = acc
            result['acc']['epoch'] = i + 1
        if f1 > result['f1']['value']:
            result['f1']['value'] = f1
            result['f1']['epoch'] = i + 1
        if pr_auc > result['pr_auc']['value']:
            result['pr_auc']['value'] = pr_auc
            result['pr_auc']['epoch'] = i + 1
            # Save the model checkpoint when PR_AUC improves
            torch.save(detector.state_dict(), DETECTOR_PATH)

    logging.info(f'Best Accuracy: {result["acc"]["value"]:.4f} at epoch {result["acc"]["epoch"]}')
    logging.info(f'Best F1: {result["f1"]["value"]:.4f} at epoch {result["f1"]["epoch"]}')
    logging.info(f'Best PR-AUC: {result["pr_auc"]["value"]:.4f} at epoch {result["pr_auc"]["epoch"]}')
    print(f'Best Accuracy: {result["acc"]["value"]:.4f} at epoch {result["acc"]["epoch"]}')
    print(f'Best F1: {result["f1"]["value"]:.4f} at epoch {result["f1"]["epoch"]}')
    print(f'Best PR-AUC: {result["pr_auc"]["value"]:.4f} at epoch {result["pr_auc"]["epoch"]}')
    
    # Clean up
    logger = logging.getLogger()
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)
    return detector, result

In [ ]:
detector, result = train(
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    hidden_size=hidden_size,
    logging_path=LOGGING_PATH,
    train_ratio=TRAIN_RATIO,
    device=DEVICE,
)

with open(OUTPUT_PATH, 'w') as f:
    json.dump(result, f, indent=2)